# Energy Demand Forecasting

Predicting hourly electricity demand for the PJM grid using time series features and gradient boosting.

Dataset: PJM Hourly Energy Consumption (Kaggle) - PJME region, ~145k hourly readings from 2002 to 2018.

> Run `python data/download.py` first to get `data/PJME_hourly.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## Load Data

In [ ]:
df = pd.read_csv('data/PJME_hourly.csv', parse_dates=['Datetime'], index_col='Datetime')
df.columns = ['MW']
df = df.sort_index()
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head(10)

In [ ]:
df.describe()

## Time Series Overview

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# full series
axes[0].plot(df.index, df['MW'], linewidth=0.4, color='steelblue', alpha=0.8)
axes[0].set_title('PJM East - Full Series (2002-2018)')
axes[0].set_ylabel('MW')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# one year zoom
year = df['2017']
axes[1].plot(year.index, year['MW'], linewidth=0.6, color='steelblue')
axes[1].set_title('2017 - Single Year')
axes[1].set_ylabel('MW')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

In [ ]:
# one week zoom - shows daily seasonality clearly
week = df['2017-07-01':'2017-07-07']
plt.figure(figsize=(12, 4))
plt.plot(week.index, week['MW'], marker='o', markersize=3, linewidth=1.2, color='steelblue')
plt.title('One Week (July 2017) - Hourly Pattern')
plt.ylabel('MW')
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%a %d'))
plt.tight_layout()
plt.show()

## Seasonality Breakdown

Three clear seasonal patterns: daily (hour of day), weekly (weekday vs weekend), annual (summer/winter peaks).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

hourly_avg = df.groupby(df.index.hour)['MW'].mean()
dow_avg = df.groupby(df.index.dayofweek)['MW'].mean()
monthly_avg = df.groupby(df.index.month)['MW'].mean()

axes[0].plot(hourly_avg.index, hourly_avg.values, marker='o', color='steelblue')
axes[0].set_title('Average by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Avg MW')

days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1].bar(days, dow_avg.values, color='steelblue')
axes[1].set_title('Average by Day of Week')
axes[1].set_ylabel('Avg MW')

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[2].bar(months, monthly_avg.values, color='steelblue')
axes[2].set_title('Average by Month')
axes[2].set_ylabel('Avg MW')

plt.tight_layout()
plt.show()

## Feature Engineering

Calendar features + lag features (past demand) + rolling statistics.

In [ ]:
def add_features(df):
    df = df.copy()
    # calendar
    df['hour'] = df.index.hour
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month
    df['quarter'] = df.index.quarter
    df['dayofyear'] = df.index.dayofyear
    df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)
    # lag features: same hour yesterday, 2 days ago, same hour last week
    df['lag_24h'] = df['MW'].shift(24)
    df['lag_48h'] = df['MW'].shift(48)
    df['lag_168h'] = df['MW'].shift(168)
    # rolling stats
    df['rolling_mean_24h'] = df['MW'].shift(1).rolling(24).mean()
    df['rolling_std_24h'] = df['MW'].shift(1).rolling(24).std()
    df['rolling_mean_168h'] = df['MW'].shift(1).rolling(168).mean()
    return df

df_feat = add_features(df)
df_feat = df_feat.dropna()
print(f"Shape after features: {df_feat.shape}")
df_feat.head()

## Train / Test Split

Split at 2017-01-01. Training on ~15 years, testing on ~1 year.

In [ ]:
SPLIT_DATE = '2017-01-01'

train = df_feat[df_feat.index < SPLIT_DATE]
test = df_feat[df_feat.index >= SPLIT_DATE]

feature_cols = [c for c in df_feat.columns if c != 'MW']
X_train = train[feature_cols]
y_train = train['MW']
X_test = test[feature_cols]
y_test = test['MW']

print(f"Train: {len(train)} rows ({train.index.min().date()} to {train.index.max().date()})")
print(f"Test:  {len(test)} rows ({test.index.min().date()} to {test.index.max().date()})")

## Scoring Helper

In [ ]:
def score(name, preds, actuals):
    mae = mean_absolute_error(actuals, preds)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mape = np.mean(np.abs((actuals - preds) / actuals)) * 100
    print(f"{name:30s}  MAE={mae:7,.0f} MW   RMSE={rmse:7,.0f} MW   MAPE={mape:.2f}%")
    return {'name': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []

## Baseline: Naive (same hour last week)

Just use lag_168h as the prediction. Surprisingly hard to beat.

In [ ]:
naive_pred = X_test['lag_168h']
results.append(score('Naive (lag 168h)', naive_pred, y_test))

## Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
results.append(score('Linear Regression', lr_pred, y_test))

## XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_pred = xgb_model.predict(X_test)
results.append(score('XGBoost', xgb_pred, y_test))

## LightGBM

In [ ]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)
results.append(score('LightGBM', lgb_pred, y_test))

## Results Comparison

In [ ]:
results_df = pd.DataFrame(results).set_index('name')
results_df = results_df.round(2)
print(results_df.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
results_df['MAPE'].plot(kind='bar', ax=ax, color='steelblue', rot=30)
ax.set_title('MAPE by Model (lower is better)')
ax.set_ylabel('MAPE (%)')
plt.tight_layout()
plt.show()

## Forecast vs Actual - 2 Week Sample

In [ ]:
sample = test['2017-07-01':'2017-07-14']
sample_idx = sample.index

plt.figure(figsize=(14, 5))
plt.plot(sample_idx, y_test[sample_idx], label='Actual', color='black', linewidth=1)
plt.plot(sample_idx, lgb_pred[test.index.isin(sample_idx)], label='LightGBM', color='steelblue', linewidth=1, alpha=0.85)
plt.plot(sample_idx, naive_pred[sample_idx], label='Naive', color='tomato', linewidth=1, linestyle='--', alpha=0.7)
plt.title('Forecast vs Actual - July 2017')
plt.ylabel('MW')
plt.legend()
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.savefig('forecast_sample.png', dpi=150)
plt.show()

## Feature Importance

In [ ]:
imp = pd.Series(lgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
imp.head(8).sort_values().plot(kind='barh', color='steelblue')
plt.title('LightGBM - Top 8 Features')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()
print(imp.head(8))

## Summary

Lag features dominate - `lag_168h` (same hour last week) is by far the most important feature. This makes sense: energy demand is highly periodic.

LightGBM edges out XGBoost slightly on this dataset. Both comfortably beat the naive and linear baselines.

Next steps worth trying: adding weather data (temperature is a huge driver of demand), holiday flags, and longer lag windows.